# atari_endpoint_replicate — DDQN vs. reference DDQN on 12 Atari games (50M frames)

One plot per game: the `ddqn_atari` component runs DDQN with a uniform 1M
replay buffer under `EPRAtariConfig`, sweeping `ENV_HYPERS.GAME` over 12
games at one seed each, with deterministic GPU kernels. Each plot shows the
run as a `tab:blue` curve, overlaid against a solid black **Endpoint Ref**
curve (see below). Atari is
cluster-scale, so this notebook expects results synced from the cluster
(`run.py sync`).

**Endpoint Ref** is the mean over 10 seeds of coresets' DDQN baseline with
a uniform 1M-size replay buffer (the `truefinal/large` config in
`coresets/experiments/atari-50M-endpoint`), interpolated onto a shared
frame grid. Cached locally as `coresets_ddqn_1m_reference.json`, whose
`_source` key records the source config and results database.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_HERE = Path.cwd()
_EXP_DIR = _HERE if (_HERE / "config.py").exists() else Path("experiments/atari_10m_deterministic")
sys.path.insert(0, str(_EXP_DIR))
sys.path.insert(0, str(_EXP_DIR.resolve().parents[1]))

from experiment import load_result, load_runs
from analysis.plotting import seed_grids_for, style

from config import EXPERIMENT

FONT_SIZE = 20   # 2x matplotlib's default (10), passed to every text call below


def grid_for(component, n=500):
    """A shared [0, total_steps] timestep grid from any run's length."""
    df = load_runs(EXPERIMENT, component)
    total_steps = len(load_result(EXPERIMENT, component, df["run_id"][0])["reward"])
    return np.linspace(0, total_steps, n)


def env_title(game):
    """The plot title for one game: e.g. "Battle Zone"."""
    return game.replace("_", " ").title()


_CORESETS_REFERENCE = json.loads(
    (_EXP_DIR / "coresets_ddqn_1m_reference.json").read_text()
)


def coresets_reference_curve(game):
    """coresets' DDQN, uniform 1M-buffer curve for `game`.

    Mean return over its 10 seeds (truefinal/large config), interpolated
    onto a shared frame grid; see coresets_ddqn_1m_reference.json.
    """
    ref = _CORESETS_REFERENCE[game]
    return np.array(ref["frames"]), np.array(ref["mean_return"])


In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else _EXP_DIR / "results"

COMPONENT = "ddqn_atari"

GRID = grid_for(COMPONENT)

# GRID is in env steps; scale to frames (steps x frameskip) for the x-axis.
FRAMESKIP = load_runs(EXPERIMENT, COMPONENT)["ENV_HYPERS.FRAMESKIP"][0]
FRAME_GRID = GRID * FRAMESKIP
FRAME_TICKS = np.array([10, 20, 30, 40, 50]) * 1_000_000
FRAME_TICK_LABELS = [f"{t // 1_000_000}M" for t in FRAME_TICKS]


In [ ]:
figures = {}
curves = {}
df = load_runs(EXPERIMENT, COMPONENT)
for game in df["ENV_HYPERS.GAME"].unique(maintain_order=True):
    run_ids = df.filter(df["ENV_HYPERS.GAME"] == game)["run_id"].to_list()
    (curve,) = seed_grids_for(EXPERIMENT, COMPONENT, GRID, run_ids=run_ids)

    fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
    ax.plot(FRAME_GRID, curve, lw=1.75, color="tab:blue", label="DDQN")
    cs_frames, cs_return = coresets_reference_curve(game)
    cs_in_range = cs_frames <= FRAME_GRID[-1]  # match our training window
    ax.plot(
        cs_frames[cs_in_range], cs_return[cs_in_range], color="black",
        lw=2.5, label="Endpoint Ref",
    )
    ax.set_title(env_title(game), fontsize=FONT_SIZE)
    ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)
    style(ax, xlabel="Frames")   # score range differs per game, so let it auto-scale
    ax.set_ylabel(
        "Return", rotation=90, ha="center", va="center",
        labelpad=20, fontsize=FONT_SIZE,
    )
    ax.set_xlabel(ax.get_xlabel(), fontsize=FONT_SIZE)
    ax.set_xticks(FRAME_TICKS)
    ax.set_xticklabels(FRAME_TICK_LABELS)
    ax.tick_params(labelsize=FONT_SIZE)
    fig.tight_layout()
    figures[f"{COMPONENT}_{game}"] = fig
    curves[game] = curve

plt.show()


In [ ]:
# (column, row) of each game's panel
GAMES_GRID = {
    "battle_zone": (0, 0),
    "double_dunk": (0, 1),
    "name_this_game": (0, 2),
    "phoenix": (1, 0),
    "qbert": (1, 1),
    "pong": (1, 2),
    "breakout": (2, 0),
    "seaquest": (2, 1),
    "centipede": (2, 2),
    "ms_pacman": (3, 0),
    "beam_rider": (3, 1),
    "space_invaders": (3, 2),
}

grid_fig, axes = plt.subplots(3, 4, figsize=(16, 9.5))
for game, (col, row) in GAMES_GRID.items():
    ax = axes[row, col]
    ax.set_box_aspect(3 / 4)   # 4:3 width:height
    ax.plot(FRAME_GRID, curves[game], lw=1.75, color="tab:blue", label="DDQN")
    cs_frames, cs_return = coresets_reference_curve(game)
    cs_in_range = cs_frames <= FRAME_GRID[-1]  # match our training window
    ax.plot(
        cs_frames[cs_in_range], cs_return[cs_in_range], color="black",
        lw=2.5, label="Endpoint Ref",
    )
    ax.set_title(env_title(game), fontsize=FONT_SIZE)
    style(ax, xlabel="", ylabel="")
    ax.set_xticks([0, FRAME_GRID[-1]])
    ax.set_xticklabels(["0", f"{FRAME_GRID[-1] / 1_000_000:.0f}M"])
    ax.tick_params(labelsize=0.7 * FONT_SIZE)
# Leave the bottom free for the Frames label with the legend under it.
grid_fig.tight_layout(rect=(0.03, 0.06, 1, 1), w_pad=0.2, h_pad=0.2)
grid_fig.supxlabel("Frames", y=0.055, va="top", fontsize=FONT_SIZE)
grid_fig.supylabel("Return", fontsize=FONT_SIZE)
grid_fig.legend(
    *axes[0, 0].get_legend_handles_labels(), loc="upper center",
    bbox_to_anchor=(0.5, 0.035), ncol=2, frameon=False, fontsize=FONT_SIZE,
)
figures["grid"] = grid_fig

plt.show()


In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")
